# NOTEBOOK 1: CARGA Y EXPLORACIÓN INICIAL DE DATOS

---

## Proyecto: Arquitectura de BI y Big Data para Análisis del Turismo Académico en Medellín

**Autor:** Proyecto de Grado - Especialización

**Fecha:** Octubre 2025

**Objetivo del Notebook:** Cargar los datos crudos de movilidad estudiantil de tres universidades (IUSH, Universidad de Antioquia, UNAC) y realizar una exploración inicial para identificar la estructura, calidad y problemas evidentes en los datos.

---

### Contenido:
1. Importación de librerías necesarias
2. Carga de datos de las 3 universidades
3. Inspección de dimensiones y estructura
4. Análisis preliminar de calidad de datos
5. Visualizaciones exploratorias básicas
6. Conclusiones y próximos pasos

---
## 1. IMPORTACIÓN DE LIBRERÍAS

In [1]:
# Librerías básicas para manipulación de datos
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Librerías para visualización
import matplotlib.pyplot as plt
import seaborn as sns

# Configuración de estilo para visualizaciones profesionales
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 10

# Para manejo de rutas de archivos
from pathlib import Path
import os

print("✓ Librerías importadas correctamente")
print(f"Versión de Pandas: {pd.__version__}")
print(f"Versión de NumPy: {np.__version__}")

✓ Librerías importadas correctamente
Versión de Pandas: 2.3.3
Versión de NumPy: 2.3.4


---
## 2. DEFINICIÓN DE RUTAS Y CONFIGURACIÓN

In [ ]:
# Definir rutas del proyecto
BASE_DIR = Path(os.getcwd())
DATA_RAW_DIR = BASE_DIR / 'data' / 'raw'
DATA_PROCESSED_DIR = BASE_DIR / 'data' / 'processed'
OUTPUTS_DIR = BASE_DIR / 'outputs' / 'graficas'

# Crear directorios si no existen
DATA_PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)

print("Directorios del proyecto:")
print(f"  - Datos crudos: {DATA_RAW_DIR}")
print(f"  - Datos procesados: {DATA_PROCESSED_DIR}")
print(f"  - Gráficas: {OUTPUTS_DIR}")

---
## 3. CARGA DE DATOS DE LAS TRES UNIVERSIDADES

Cargaremos los archivos CSV de cada universidad. Los datos pueden contener:
- Filas vacías (con valores #N/A)
- Formatos inconsistentes
- Valores nulos

Usaremos parámetros específicos para manejar estos problemas durante la carga inicial.

In [ ]:
def cargar_datos_universidad(archivo_path, nombre_universidad):
    """
    Carga los datos de una universidad desde un archivo CSV.
    
    Args:
        archivo_path (Path): Ruta del archivo CSV
        nombre_universidad (str): Nombre de la universidad
    
    Returns:
        DataFrame: Datos cargados con columna adicional de universidad
    """
    print(f"\n{'='*60}")
    print(f"Cargando datos de: {nombre_universidad}")
    print(f"{'='*60}")
    
    try:
        # Cargar datos, tratando #N/A como NaN
        df = pd.read_csv(
            archivo_path,
            na_values=['#N/A', 'N/A', 'NA', '', ' '],
            keep_default_na=True,
            encoding='utf-8'
        )
        
        # Agregar columna identificadora de universidad
        df['UNIVERSIDAD'] = nombre_universidad
        
        # Información básica
        print(f"✓ Archivo cargado exitosamente")
        print(f"  - Dimensiones: {df.shape[0]} filas × {df.shape[1]} columnas")
        print(f"  - Columnas: {list(df.columns)}")
        
        return df
        
    except FileNotFoundError:
        print(f"✗ ERROR: No se encontró el archivo {archivo_path}")
        return None
    except Exception as e:
        print(f"✗ ERROR al cargar datos: {str(e)}")
        return None

In [ ]:
# Cargar datos de IUSH (datos reales)
df_iush = cargar_datos_universidad(
    DATA_RAW_DIR / 'iush.csv',
    'IUSH'
)

# Cargar datos de Universidad de Antioquia (cuando estén disponibles)
df_udea = cargar_datos_universidad(
    DATA_RAW_DIR / 'universidad_antioquia.csv',
    'Universidad de Antioquia'
)

# Cargar datos de UNAC (cuando estén disponibles)
df_unac = cargar_datos_universidad(
    DATA_RAW_DIR / 'unac.csv',
    'UNAC'
)

---
## 4. INSPECCIÓN DETALLADA DE DATOS - IUSH

Realizaremos un análisis detallado del dataset de IUSH (datos reales disponibles).

In [ ]:
if df_iush is not None:
    print("\n" + "="*80)
    print("EXPLORACIÓN DETALLADA - IUSH")
    print("="*80)
    
    # 1. Información general
    print("\n1. INFORMACIÓN GENERAL DEL DATASET")
    print("-" * 80)
    print(f"Dimensiones: {df_iush.shape[0]:,} filas × {df_iush.shape[1]} columnas")
    print(f"\nTipos de datos:")
    print(df_iush.dtypes)
    
    # 2. Primeras filas
    print("\n2. PRIMERAS 5 FILAS DEL DATASET")
    print("-" * 80)
    display(df_iush.head())
    
    # 3. Estadísticas descriptivas básicas
    print("\n3. ESTADÍSTICAS DESCRIPTIVAS")
    print("-" * 80)
    display(df_iush.describe(include='all').T)
    
    # 4. Valores nulos
    print("\n4. ANÁLISIS DE VALORES NULOS")
    print("-" * 80)
    nulos = df_iush.isnull().sum()
    porcentaje_nulos = (nulos / len(df_iush)) * 100
    
    resumen_nulos = pd.DataFrame({
        'Cantidad_Nulos': nulos,
        'Porcentaje': porcentaje_nulos
    }).sort_values('Cantidad_Nulos', ascending=False)
    
    display(resumen_nulos[resumen_nulos['Cantidad_Nulos'] > 0])
    
    # 5. Filas completamente vacías
    filas_vacias = df_iush.isnull().all(axis=1).sum()
    print(f"\nFilas completamente vacías: {filas_vacias:,}")
    print(f"Porcentaje de filas vacías: {(filas_vacias/len(df_iush)*100):.2f}%")
else:
    print("✗ No se pudieron cargar los datos de IUSH")

---
## 5. IDENTIFICACIÓN DE PROBLEMAS DE CALIDAD

Identificaremos los principales problemas de calidad de datos que deberemos resolver en el siguiente notebook.

In [ ]:
if df_iush is not None:
    print("\n" + "="*80)
    print("PROBLEMAS DE CALIDAD IDENTIFICADOS")
    print("="*80)
    
    problemas = []
    
    # Eliminar filas completamente vacías para análisis
    df_iush_limpio_temp = df_iush.dropna(how='all')
    
    # 1. Filas vacías
    filas_vacias = len(df_iush) - len(df_iush_limpio_temp)
    if filas_vacias > 0:
        problemas.append(f"✗ {filas_vacias:,} filas completamente vacías detectadas")
    
    # 2. Valores nulos en columnas importantes
    columnas_importantes = ['AÑO', 'SEMESTRE', 'PAIS_EXTRANJERO', 'TIPO_MOV_EST_EXTRANJ']
    for col in columnas_importantes:
        if col in df_iush_limpio_temp.columns:
            nulos = df_iush_limpio_temp[col].isnull().sum()
            if nulos > 0:
                problemas.append(f"✗ {nulos} valores nulos en columna '{col}'")
    
    # 3. Duplicados potenciales
    duplicados = df_iush_limpio_temp.duplicated().sum()
    if duplicados > 0:
        problemas.append(f"✗ {duplicados} registros duplicados detectados")
    
    # 4. Espacios en blanco en nombres
    for col in ['PRIMER_NOMBRE', 'PRIMER_APELLIDO', 'PAIS_EXTRANJERO']:
        if col in df_iush_limpio_temp.columns:
            con_espacios = df_iush_limpio_temp[col].astype(str).str.strip().ne(
                df_iush_limpio_temp[col].astype(str)
            ).sum()
            if con_espacios > 0:
                problemas.append(f"✗ {con_espacios} registros con espacios adicionales en '{col}'")
    
    # 5. Consistencia de países
    if 'PAIS_EXTRANJERO' in df_iush_limpio_temp.columns:
        paises_unicos = df_iush_limpio_temp['PAIS_EXTRANJERO'].nunique()
        problemas.append(f"ℹ {paises_unicos} países únicos detectados (verificar nomenclatura)")
    
    # Imprimir problemas
    for i, problema in enumerate(problemas, 1):
        print(f"{i}. {problema}")
    
    print(f"\n{'='*80}")
    print(f"TOTAL DE PROBLEMAS IDENTIFICADOS: {len(problemas)}")
    print(f"{'='*80}")

---
## 6. VISUALIZACIONES EXPLORATORIAS BÁSICAS

Crearemos visualizaciones preliminares para entender mejor la distribución de los datos.

In [ ]:
if df_iush is not None:
    # Trabajar solo con datos no vacíos
    df_viz = df_iush.dropna(how='all')
    
    print(f"\nDatos para visualización: {len(df_viz):,} registros válidos")
    
    # Crear figura con múltiples subplots
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    fig.suptitle('Exploración Inicial de Datos - IUSH', fontsize=16, fontweight='bold')
    
    # 1. Distribución por país de origen
    if 'PAIS_EXTRANJERO' in df_viz.columns:
        top_paises = df_viz['PAIS_EXTRANJERO'].value_counts().head(10)
        top_paises.plot(kind='barh', ax=axes[0, 0], color='steelblue')
        axes[0, 0].set_title('Top 10 Países de Origen de Estudiantes', fontweight='bold')
        axes[0, 0].set_xlabel('Cantidad de Estudiantes')
        axes[0, 0].set_ylabel('País')
    
    # 2. Distribución por tipo de movilidad
    if 'TIPO_MOV_EST_EXTRANJ' in df_viz.columns:
        tipo_mov = df_viz['TIPO_MOV_EST_EXTRANJ'].value_counts()
        tipo_mov.plot(kind='bar', ax=axes[0, 1], color='coral')
        axes[0, 1].set_title('Distribución por Tipo de Movilidad', fontweight='bold')
        axes[0, 1].set_xlabel('Tipo de Movilidad')
        axes[0, 1].set_ylabel('Cantidad')
        axes[0, 1].tick_params(axis='x', rotation=45)
    
    # 3. Distribución de duración de estadía (en días)
    if 'NUM_DIAS_MOVILIDAD' in df_viz.columns:
        dias_validos = df_viz['NUM_DIAS_MOVILIDAD'].dropna()
        if len(dias_validos) > 0:
            axes[1, 0].hist(dias_validos, bins=20, color='seagreen', edgecolor='black')
            axes[1, 0].set_title('Distribución de Duración de Movilidad (días)', fontweight='bold')
            axes[1, 0].set_xlabel('Días')
            axes[1, 0].set_ylabel('Frecuencia')
            axes[1, 0].axvline(dias_validos.mean(), color='red', linestyle='--', 
                              label=f'Media: {dias_validos.mean():.1f} días')
            axes[1, 0].legend()
    
    # 4. Completitud de datos por columna (top 10 con más nulos)
    nulos_por_col = df_viz.isnull().sum().sort_values(ascending=False).head(10)
    porcentaje_nulos = (nulos_por_col / len(df_viz)) * 100
    porcentaje_nulos.plot(kind='barh', ax=axes[1, 1], color='indianred')
    axes[1, 1].set_title('Top 10 Columnas con Más Valores Nulos', fontweight='bold')
    axes[1, 1].set_xlabel('Porcentaje de Nulos (%)')
    axes[1, 1].set_ylabel('Columna')
    
    plt.tight_layout()
    
    # Guardar gráfica
    plt.savefig(OUTPUTS_DIR / '01_exploracion_inicial_iush.png', dpi=300, bbox_inches='tight')
    print(f"\n✓ Gráfica guardada en: {OUTPUTS_DIR / '01_exploracion_inicial_iush.png'}")
    
    plt.show()

---
## 7. RESUMEN DE DATOS VÁLIDOS

In [ ]:
if df_iush is not None:
    df_validos = df_iush.dropna(how='all')
    
    print("\n" + "="*80)
    print("RESUMEN DE DATOS VÁLIDOS - IUSH")
    print("="*80)
    
    print(f"\nRegistros totales en archivo: {len(df_iush):,}")
    print(f"Registros válidos (no vacíos): {len(df_validos):,}")
    print(f"Registros a eliminar: {len(df_iush) - len(df_validos):,}")
    print(f"\nPorcentaje de datos válidos: {(len(df_validos)/len(df_iush)*100):.2f}%")
    
    if 'AÑO' in df_validos.columns:
        print(f"\nAños presentes en los datos: {sorted(df_validos['AÑO'].dropna().unique())}")
    
    if 'SEMESTRE' in df_validos.columns:
        print(f"Semestres: {sorted(df_validos['SEMESTRE'].dropna().unique())}")
    
    if 'PAIS_EXTRANJERO' in df_validos.columns:
        print(f"\nCantidad de países únicos: {df_validos['PAIS_EXTRANJERO'].nunique()}")
        print(f"Principales países: {list(df_validos['PAIS_EXTRANJERO'].value_counts().head(5).index)}")

---
## 8. CONCLUSIONES Y PRÓXIMOS PASOS

### Hallazgos Principales:

1. **Calidad de Datos:**
   - Se identificaron múltiples filas completamente vacías que deben eliminarse
   - Existen valores nulos en columnas importantes que requieren tratamiento
   - Se detectaron posibles inconsistencias en nomenclatura de países

2. **Estructura de Datos:**
   - El dataset de IUSH contiene información sobre movilidad estudiantil entrante
   - Incluye datos demográficos, institucionales y de financiación
   - La duración de movilidad está registrada en días

3. **Cobertura:**
   - Actualmente solo tenemos datos de IUSH
   - Se necesitan datos de Universidad de Antioquia y UNAC para completar el análisis

### Próximos Pasos (Notebook 2):

1. **Limpieza de Datos:**
   - Eliminar filas completamente vacías
   - Tratar valores nulos según estrategia definida
   - Eliminar duplicados

2. **Estandarización:**
   - Normalizar nombres de países
   - Estandarizar nombres de columnas
   - Convertir tipos de datos apropiados

3. **Consolidación:**
   - Integrar datos de las 3 universidades cuando estén disponibles
   - Crear variables derivadas útiles para el análisis
   - Exportar dataset limpio para análisis posterior

---
**Fin del Notebook 1**

Continuar con: `02_limpieza_estandarizacion_etl.ipynb`